In [ ]:
import os

In [ ]:
os.chdir("../")

In [ ]:

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import rdmolops
from torch_geometric.data import Data, Dataset
from dataclasses import dataclass
from pathlib import Path
from src.GNNClassfier.constants import *
from src.GNNClassfier.utils.common import read_yaml, create_directories
from src.GNNClassfier import logger

In [ ]:
@dataclass
class DataPreparationConfig:
    root_dir: Path
    train_csv_path: Path
    test_csv_path: Path
    train_graph_dir: Path
    test_graph_dir: Path

In [ ]:
# 2. Updated Configuration Manager
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_preparation_config(self) -> DataPreparationConfig:
        config = self.config.data_preparation
        trans_config = self.config.data_transformation
        
        create_directories([config.root_dir, config.train_graph_dir, config.test_graph_dir])

        return DataPreparationConfig(
            root_dir=Path(config.root_dir),
            train_csv_path=Path(trans_config.train_file),
            test_csv_path=Path(trans_config.test_file),
            train_graph_dir=Path(config.train_graph_dir),
            test_graph_dir=Path(config.test_graph_dir)
        )

In [ ]:
# 3. Graph Construction Logic (The Component)
class MoleculeGraphGenerator:
    def __init__(self, config: DataPreparationConfig):
        self.config = config

    def generate_graphs(self, csv_path: Path, output_dir: Path, set_name: str):
        df = pd.read_csv(csv_path)
        logger.info(f"Generating graphs for {set_name} set ({len(df)} molecules)...")
        
        # Ensure processed sub-directory exists (PyTorch Geometric requirement)
        processed_dir = output_dir / "processed"
        os.makedirs(processed_dir, exist_ok=True)

        for i, row in tqdm(enumerate(df.itertuples()), total=len(df), desc=f"Processing {set_name}"):
            mol_obj = Chem.MolFromSmiles(row.smiles)
            if mol_obj is None: continue

            node_features = self._get_node_features(mol_obj)
            edge_features = self._get_edge_features(mol_obj)
            edge_index = self._get_adjacency_info(mol_obj)
            label = torch.tensor([row.HIV_active], dtype=torch.int64)

            data = Data(x=node_features, 
                        edge_index=edge_index, 
                        edge_attr=edge_features, 
                        y=label, 
                        smiles=row.smiles)

            torch.save(data, os.path.join(processed_dir, f"data_{i}.pt"))

    def _get_node_features(self, mol):
        all_node_features = []
        for atom in mol.GetAtoms():
            node_feats = [
                atom.GetAtomicNum(),
                atom.GetDegree(),
                atom.GetFormalCharge(),
                int(atom.GetHybridization()),
                int(atom.GetIsAromatic()),
                atom.GetTotalNumHs()
            ]
            all_node_features.append(node_feats)
        return torch.tensor(np.array(all_node_features), dtype=torch.float)

    def _get_edge_features(self, mol):
        all_edge_features = []
        for bond in mol.GetBonds():
            edge_feats = [bond.GetBondTypeAsDouble(), int(bond.IsInRing())]
            all_edge_features.append(edge_feats)
            # Add reverse edge features for undirected graphs
            all_edge_features.append(edge_feats)
        return torch.tensor(np.array(all_edge_features), dtype=torch.float)

    def _get_adjacency_info(self, mol):
        adj_matrix = rdmolops.GetAdjacencyMatrix(mol)
        row, col = np.where(adj_matrix != 0)
        edge_index = torch.tensor(np.array([row, col]), dtype=torch.long)
        return edge_index

In [ ]:
# 4. Pipeline Execution
try:
    config_manager = ConfigurationManager()
    prep_config = config_manager.get_data_preparation_config()
    generator = MoleculeGraphGenerator(config=prep_config)
    
    # Process Training Set
    generator.generate_graphs(prep_config.train_csv_path, prep_config.train_graph_dir, "Training")
    
    # Process Test Set
    generator.generate_graphs(prep_config.test_csv_path, prep_config.test_graph_dir, "Test")
    
except Exception as e:
    logger.exception(e)
    raise e